# Outbound disclosure screening

**Recipe 42** · Level 4 (Advanced) · Decision type: `Noul`

Screen proposed outbound messages for disclosure concerns against supplied sharing rules while deterministic controls hold flagged or uncertain samples for review.

Sources:

- S02: [TypeSafe AI: Primitives (Questions)](https://docs.typesafe.ai/primitives)
- S03: [TypeSafe AI: Confidence](https://docs.typesafe.ai/confidence)
- S04: [TypeSafe AI: How to build with TypeSafe](https://docs.typesafe.ai/concepts/how-to-build-with-system-one)
- S07: [TypeSafe AI: Jev 1.13 jaggedness](https://docs.typesafe.ai/model-jaggedness/jev-1.13)

## What you will build

A screening pass over 35 synthetic outbound messages from a fictional vendor, Meridian Robotics (every name, number and address fabricated). Two sharing rules are supplied: does the message state unpublished pricing, and does it disclose a specific person's contact details. Jev answers both as independent `Noul` propositions, asked together in one request per message. Python owns everything downstream: a business threshold and a confidence gate per rule (recipe 15's three-path pattern, pooled across both rules), a deterministic policy that holds every later message in a thread once an earlier one in it was held, and the final `held`/`allowed` decision. Nothing is ever sent: `allowed` only means a message cleared every check in this recipe's simulation, logged with `jev_cookbook.simulation.ActionLog`, which executes nothing.

## Setup and run mode

The notebook runs from its own folder with the standard library and `jev_cookbook` only. `get_backend(fixtures=...)` replays the 35 stored answers in `fixtures/responses.json` offline; setting `JEV_COOKBOOK_LIVE=1` (see the README) swaps in live calls to the same two questions and changes nothing else. `run_header` states which mode ran, and every number below carries that mode with it.

In [ ]:
from jev_cookbook import get_backend, load_helpers
from jev_cookbook.evaluation import (
    evaluate_selective,
    evaluate_threshold,
    pool_label_decisions,
    select_confidence_threshold,
    select_threshold,
    selective_curve,
)
from jev_cookbook.fixtures import load_inputs, load_labels, responses_path, select_split
from jev_cookbook.simulation import ActionLog, ReviewQueue
from jev_cookbook.style import (
    apply_style,
    plot_answer_probabilities,
    plot_confusion_matrix,
    plot_risk_coverage,
    run_header,
    show_answer,
)

apply_style()
helpers = load_helpers()  # this recipe's helpers.py, loaded by path
examples = load_inputs()
labels = load_labels()
backend = get_backend(fixtures=responses_path())
questions = helpers.build_questions()
# N for the header is the number of messages the metrics are about: validation and test. This
# recipe has no train split; the 3 demo messages are shown but never scored.
scored = [e for e in examples if e.split not in ("train", "demo")]

offline = backend.mode in ("synthetic", "scripted")
check = " (a pipeline check, not a Jev result)" if offline else ""
# CONTRIBUTING.md section 2: a number from the validation split is a selection step, not a
# result. A validation number always carries this label; in an offline run it also carries
# `check`, concatenated after it.
selection = " (a selection step, not a reported result)"

# Every message needs at most one request -- the two-question sharing-rule battery -- however
# many times it is shown or scored below. This cache is what keeps that true: it is read by
# every later cell, so the live budget this notebook spends equals exactly the number of
# distinct messages in fixtures/, never more.
answered = {}


def decide(example):
    if example.id not in answered:
        answered[example.id] = backend.decide(helpers.build_state(example.fields), questions)
    return answered[example.id]


run_header(42, "Outbound disclosure screening", backend=backend, n_examples=len(scored))

## The state

The state is everything Jev sees, and Python builds it. Each example in `fixtures/inputs.jsonl` carries `fields`: the proposed message `text`, plus a `thread_id` and a `sequence` number. `build_state` passes on the text only; the thread bookkeeping stays in Python for the stateful policy further down, so it never passes through the model and two messages with the same text but different thread bookkeeping would share one replay key.

In [ ]:
import json

demo = {e.id: e for e in examples if e.split == "demo"}
example = demo["d01-held"]
print("Python has:")
print(json.dumps(example.fields, indent=2))
state = helpers.build_state(example.fields)
print("Jev sees:")
print(json.dumps(state, indent=2))

## The questions

Two sharing rules, two independent propositions, sent together in one request for every message (CONTRIBUTING.md section 3: "Independent questions go in one request"). Each is written as a statement that can be true or false ([S02](https://docs.typesafe.ai/primitives)), not as a question: `pricing` asks whether the message states a specific, unpublished price or deal term; `contact` asks whether it discloses a specific person's contact details or an internal identifier. Python supplies both the statement and the criteria that define `true` and `false`; Jev never sees the rule names `pricing`/`contact` below, only the text each one carries.

In [ ]:
for name, question in questions.items():
    print(name)
    print(" ", question.instructions)
    for option, description in question.criteria.items():
        print(f"   {option}: {description}")

## One answer up close

Before any aggregate, look at three typed batteries: a message that violates both sharing rules, one that violates neither, and one where an embedded instruction tries to talk its own reviewer into clearing it. Each answer in a battery holds just the probability that its own rule's statement holds, and a `provenance` line that says where it came from -- here always `synthetic`, since nothing in this recipe's fixtures came from a real call.

In [ ]:
held_example = demo["d01-held"]
held_result = decide(held_example)
print(held_example.fields["text"])
for name in helpers.RULES:
    show_answer(held_result[name])

In [ ]:
plot_answer_probabilities(held_result[helpers.PRICING], title="Pricing rule, one message up close")

This renewal note names a discount that is not on Meridian's published price list, and gives the account lead's direct number in the same breath. Both stored probabilities sit near the top of the scale (0.92 and 0.88), so both rules will treat it the same way once the business threshold and confidence gate below are chosen: held, with nothing for a person to double-check.

In [ ]:
allowed_example = demo["d02-allowed"]
allowed_result = decide(allowed_example)
print(allowed_example.fields["text"])
for name in helpers.RULES:
    show_answer(allowed_result[name])

This migration update names no figure and no person; it points the reader to Meridian's own published pricing page instead of stating a number. Both stored probabilities sit near the bottom of the scale (0.04 and 0.03): a message can be entirely routine and never come near either sharing rule.

In [ ]:
adversarial_example = demo["d03-adversarial"]
adversarial_result = decide(adversarial_example)
print(adversarial_example.fields["text"])
for name in helpers.RULES:
    show_answer(adversarial_result[name])

This message opens with a line addressed to whoever reviews outbound mail, instructing it to skip the check -- exactly the kind of adversarial content TypeSafe's own jaggedness notes warn about ([S07](https://docs.typesafe.ai/model-jaggedness/jev-1.13)) -- and then states a specific, unpublished renewal figure anyway. The gold label for `pricing` is true regardless of the embedded instruction. The stored answer, though, calls it confidently false at a probability of 0.08: far enough from an even split that the confidence gate below will not catch it either, because confidence measures how far a probability leans, not whether it leans the right way. The evaluation further down counts this one as wrong, on purpose: a fixture set where every stored answer is right would not teach anything about where this recipe's rule can be fooled.

## Python's part

Jev supplies two probabilities per message; what happens with them is code. `helpers.verdict_for_rule` decides one rule's own outcome -- `violates`, `clear`, or `review` -- the same three-path shape recipe 15 uses for a single Noul: the [confidence gate](../../docs/glossary.md#confidence-gate) (`noul_confidence`, "The questions" above) is checked first, whichever way the probability leans, and only once it clears does the [business threshold](../../docs/glossary.md#business-threshold) decide `violates` or `clear`. `helpers.screen_message` then composes both rules' verdicts into one message-level decision: any confident violation holds the message outright; failing that, any rule still uncertain holds it for [review](../../docs/glossary.md#review); only a message every rule confidently clears is `allowed`. Both cut-offs are chosen here, on `validation`, and then frozen for the rest of this notebook.

The business threshold is chosen per rule (two independent `select_threshold` calls, each by F1 on its own rule's gold labels); the confidence gate is chosen once, shared by both rules, from every (message, rule) pair pooled together with `jev_cookbook.evaluation.pool_label_decisions` (docs/evaluation.md, "Multi-label") -- the same multi-label pattern a per-label Noul battery always needs before `selective_curve` or `select_confidence_threshold` can see it.

In [ ]:
val_examples = select_split(examples, "validation")
val_answers = {e.id: decide(e) for e in val_examples}
val_labels = {e.id: labels[e.id] for e in val_examples}

threshold = {}
for rule in helpers.RULES:
    gold_rule = [val_labels[e.id][rule] for e in val_examples]
    noul_rule = [val_answers[e.id][rule].noul for e in val_examples]
    threshold[rule] = select_threshold(gold_rule, noul_rule, objective="f1")
    print(
        f"{rule} business threshold, frozen from here on: {threshold[rule]:.2f}{selection}{check}"
    )

# Pool every (message, rule) pair on validation -- 16 messages x 2 rules = 32 pairs, label-then-
# example (docs/evaluation.md): every message's pricing decision, then every message's contact
# decision.
val_ids = [e.id for e in val_examples]
gold_by_label = {rule: [val_labels[i][rule] for i in val_ids] for rule in helpers.RULES}
noul_by_label = {rule: [val_answers[i][rule].noul for i in val_ids] for rule in helpers.RULES}
correct, confidence, pooled_keys = pool_label_decisions(
    val_ids, list(helpers.RULES), gold_by_label, noul_by_label, threshold
)
print(f"pooled (message, rule) pairs on validation: {len(pooled_keys)}{selection}{check}")
print(
    f"  pooling order check: first key {pooled_keys[0]}, pair {len(val_ids)} is {pooled_keys[len(val_ids)]}{selection}{check}"
)

# A coverage floor, not a target accuracy: never review more than 25% of the pooled pairs
# outright, and pick the most accurate gate within that budget. The next cell explains why.
min_confidence = select_confidence_threshold(correct, confidence, min_coverage=0.75)
print(f"shared confidence gate, frozen from here on: {min_confidence:.2f}{selection}{check}")

for shown_example, shown_result in (
    (held_example, held_result),
    (allowed_example, allowed_result),
    (adversarial_example, adversarial_result),
):
    verdicts = tuple(
        helpers.verdict_for_rule(rule, shown_result[rule].noul, threshold[rule], min_confidence)
        for rule in helpers.RULES
    )
    outcome = helpers.screen_message(shown_example.id, verdicts, thread_escalated=False)
    print(f"{outcome.message_id}: {outcome.outcome} ({outcome.reason}){check}")
    for v in verdicts:
        print(f"    {v.rule}: noul {v.noul:.2f} -> {v.verdict}{check}")

### The confidence gate is a two-sided band, not a floor

`confidence = |2n - 1|` is symmetric about 0.5, so the gate passes exactly when `n >= (1 + g) / 2` (confidently true) or `n <= (1 - g) / 2` (confidently false); everything in between is `review`. With the shared gate above, `g = 0.14`, that band is `(0.43, 0.57)`. Whether the gate actually *changes* anything depends on where a rule's own business threshold sits relative to that band, not on the band alone, so the honest way to check is to measure it against this recipe's own rule, not to reason about the algebra in the abstract. The cell below sweeps 401 evenly spaced noul values from 0 to 1 through `verdict_for_rule`, once with the frozen gate and once with no gate at all (`min_confidence=0.0`), and counts how many of those 401 outcomes the gate actually changes -- split by which side of the business threshold they started on.

In [ ]:
def gate_effect(rule_name, business_threshold, gate, n_points=401):
    """Count how many of n_points evenly spaced noul values the frozen gate turns into a
    review, split by which side of business_threshold they sit on without the gate. Measures
    verdict_for_rule directly -- no algebra, no assumption about which side (if either) the
    gate is inert on."""
    accept_changed, reject_changed = 0, 0
    for i in range(n_points):
        n = i / (n_points - 1)
        ungated = helpers.verdict_for_rule(rule_name, n, business_threshold, 0.0)
        gated = helpers.verdict_for_rule(rule_name, n, business_threshold, gate)
        if gated.verdict != ungated.verdict:
            if ungated.verdict == helpers.VIOLATES:
                accept_changed += 1
            else:
                reject_changed += 1
    return accept_changed, reject_changed


lower, upper = (1 - min_confidence) / 2, (1 + min_confidence) / 2
print(f"gate band: review whenever {lower:.4f} < n < {upper:.4f}{check}")
for rule in helpers.RULES:
    accept_changed, reject_changed = gate_effect(rule, threshold[rule], min_confidence)
    print(
        f"{rule} (threshold {threshold[rule]:.2f}): "
        f"{accept_changed} of 401 accept-side outcomes changed, "
        f"{reject_changed} of 401 reject-side outcomes changed{check}"
    )

# One concrete counterfactual per side that actually changes something, printed rather than
# only counted: contact's own threshold (0.50) sits inside the band, so both sides move.
for n in (0.45, 0.50, 0.55):
    without_gate = helpers.verdict_for_rule("contact", n, threshold["contact"], 0.0)
    with_gate = helpers.verdict_for_rule("contact", n, threshold["contact"], min_confidence)
    print(
        f"  contact, n={n:.2f}: without the gate -> {without_gate.verdict}, with it -> {with_gate.verdict}{check}"
    )

### A held thread

The build note for this recipe asks for stateful handling: once a thread contains a held message, Python holds every later message in that same thread outright, whatever its own content says. `helpers.screen_thread` carries that one boolean forward across a thread's messages, in `sequence` order; `v13-thread` and `v14-thread` are this recipe's committed walkthrough of it -- two validation messages sharing one `thread_id`. `v13-thread` gives a direct billing contact's number (`contact` violates); `v14-thread` is a routine follow-up in the same thread with no pricing or contact detail of its own (both rules confidently clear). The cell below replays both, in order, and shows `v14-thread` held anyway.

In [ ]:
thread_examples = [e for e in val_examples if e.fields["thread_id"] == "v-thread-a"]
thread_examples.sort(key=lambda e: e.fields["sequence"])
thread_entries = [
    (e.id, {rule: decide(e)[rule].noul for rule in helpers.RULES}) for e in thread_examples
]
thread_results = helpers.screen_thread(thread_entries, threshold, min_confidence)
for example, result in zip(thread_examples, thread_results, strict=True):
    print(f"{result.message_id} (sequence {example.fields['sequence']}): {example.fields['text']}")
    print(f"  -> {result.outcome} ({result.reason}){check}")

## Evaluation

Four different quantities appear below, and they are not the same size: 32 scored **input rows** (16 `validation`, 16 `test`), one **request** per row (the two-question battery, what the live budget bills and what `len(responses.json)` counts), 64 scored **questions** (rows x 2 rules -- what a per-rule precision or the pooled confidence-gate numbers are over), and 32 **task-level units** (one `held`/`allowed` decision per message, what the three message-level metrics are over). Both cut-offs were chosen and frozen above, on `validation`; this section applies them to `test` and reports once. In an offline run every `test` number here is a check that the pipeline works, and says so; every `validation` number, in every run mode, is a selection step rather than a reported result, which is why it carries a different label.

In [ ]:
test_examples = select_split(examples, "test")
test_answers = {e.id: decide(e) for e in test_examples}
test_labels = {e.id: labels[e.id] for e in test_examples}
test_ids = [e.id for e in test_examples]

print(
    f"input rows: {len(val_examples)} validation + {len(test_examples)} test = {len(val_examples) + len(test_examples)}{check}"
)
print(
    f"requests: one per message -- {len(val_examples)} validation + {len(test_examples)} test{check}"
)
print(
    f"questions: {len(helpers.RULES)} rules x rows -- {len(val_examples) * 2} validation + {len(test_examples) * 2} test{check}"
)
print(
    f"task-level units: one decision per message -- {len(val_examples)} validation + {len(test_examples)} test{check}"
)

for rule in helpers.RULES:
    gold_rule = [test_labels[i][rule] for i in test_ids]
    noul_rule = [test_answers[i][rule].noul for i in test_ids]
    point = evaluate_threshold(gold_rule, noul_rule, threshold[rule])
    print(
        f"{rule}, business rule alone (ignores the gate), test: "
        f"precision {point.precision:.3f}  recall {point.recall:.3f}  f1 {point.f1:.3f}{check}"
    )

`evaluate_selective` reports what the confidence gate buys once it is applied to the pooled (message, rule) pairs -- [coverage](../../docs/glossary.md#coverage) (the share decided automatically), accuracy, and [risk](../../docs/glossary.md#risk) among those -- the confidence-only view described in docs/evaluation.md ("Outcomes versus the confidence-only view"). This is exactly right at the per-rule, per-pair level: within one rule's own verdict, the gate is the *only* review branch. It is not yet the message-level story, because `screen_message` can also hold a message that the gate alone would have let through (the other rule confidently violating, or thread escalation) -- the next section builds that from `screen_message`'s own outcomes instead.

In [ ]:
test_gold_by_label = {rule: [test_labels[i][rule] for i in test_ids] for rule in helpers.RULES}
test_noul_by_label = {
    rule: [test_answers[i][rule].noul for i in test_ids] for rule in helpers.RULES
}
test_correct, test_confidence, test_keys = pool_label_decisions(
    test_ids, list(helpers.RULES), test_gold_by_label, test_noul_by_label, threshold
)

val_curve = selective_curve(correct, confidence)
print(f"pooled (message, rule) pairs, validation, {len(correct)} pairs{selection}{check}")
for t, cov, acc, risk in zip(
    val_curve.thresholds, val_curve.coverage, val_curve.accuracy, val_curve.risk, strict=True
):
    marker = " <- frozen" if abs(t - min_confidence) < 1e-9 else ""
    print(
        f"  gate >= {t:.2f}: coverage {cov:.3f}, accuracy {acc:.3f}, "
        f"risk {risk:.3f}{marker}{selection}{check}"
    )

val_selective = evaluate_selective(correct, confidence, min_confidence)
test_selective = evaluate_selective(test_correct, test_confidence, min_confidence)
print(
    f"pooled (message, rule) pairs, validation at the frozen gate: coverage "
    f"{val_selective.coverage:.3f} accuracy {val_selective.accuracy:.3f} "
    f"risk {val_selective.risk:.3f}{selection}{check}"
)
print(
    f"pooled (message, rule) pairs, test: coverage {test_selective.coverage:.3f} "
    f"accuracy {test_selective.accuracy:.3f} risk {test_selective.risk:.3f}{check}"
)
plot_risk_coverage(
    val_curve,
    reference_risk=1 - (sum(correct) / len(correct)),
    title=f"Pooled confidence gate, validation{check}",
)

Message-level: this recipe's own three metrics, from `helpers.screening_metrics`, computed against each message's own content (`content_violates`: does either rule's [gold label](../../docs/glossary.md#gold-label) say this message violates something on its own, regardless of any thread it is in). `missed_disclosure_rate` is the share of genuinely violating messages that were `allowed` anyway; `false_hold_rate` is the share of genuinely clean messages that were `held` anyway, for any reason; `hold_rate` is the share of every scored message that was held. Reported alongside a baseline that uses the identical two business thresholds and the identical 16 requests but neither the confidence gate nor thread escalation (`min_confidence=0.0`, every message screened with `thread_escalated=False`) -- both variants spend exactly the same live budget, since neither the gate nor the escalation changes which questions are asked, only what Python does with the answers already in hand.

In [ ]:
def screen_full(split_examples):
    by_thread = {}
    for e in split_examples:
        by_thread.setdefault(e.fields["thread_id"], []).append(e)
    for group in by_thread.values():
        group.sort(key=lambda e: e.fields["sequence"])
    results = {}
    for group in by_thread.values():
        entries = [(e.id, {rule: decide(e)[rule].noul for rule in helpers.RULES}) for e in group]
        for result in helpers.screen_thread(entries, threshold, min_confidence):
            results[result.message_id] = result
    return results


def screen_baseline(split_examples):
    results = {}
    for e in split_examples:
        verdicts = tuple(
            helpers.verdict_for_rule(rule, decide(e)[rule].noul, threshold[rule], 0.0)
            for rule in helpers.RULES
        )
        results[e.id] = helpers.screen_message(e.id, verdicts, thread_escalated=False)
    return results


test_full = screen_full(test_examples)
test_baseline = screen_baseline(test_examples)
test_content_violates = [test_labels[i]["pricing"] or test_labels[i]["contact"] for i in test_ids]
full_held = [test_full[i].outcome == helpers.HELD for i in test_ids]
base_held = [test_baseline[i].outcome == helpers.HELD for i in test_ids]
m_full = helpers.screening_metrics(test_content_violates, full_held)
m_base = helpers.screening_metrics(test_content_violates, base_held)

missed_full = sum(1 for cv, h in zip(test_content_violates, full_held, strict=True) if cv and not h)
false_held_full = sum(
    1 for cv, h in zip(test_content_violates, full_held, strict=True) if not cv and h
)
print(f"test, full pipeline (gate + thread escalation), 16 requests{check}")
print(
    f"  missed-disclosure rate: {m_full.missed_disclosure_rate:.3f} ({missed_full} of {m_full.n_content_violates} genuinely violating messages){check}"
)
print(
    f"  false-hold rate: {m_full.false_hold_rate:.3f} ({false_held_full} of {m_full.n_content_clear} genuinely clean messages held anyway){check}"
)
print(f"  hold rate: {m_full.hold_rate:.3f} ({m_full.n_held} of {m_full.n_total} messages){check}")
print(f"test, business threshold alone, 16 requests{check}")
print(f"  missed-disclosure rate: {m_base.missed_disclosure_rate:.3f}{check}")
print(f"  false-hold rate: {m_base.false_hold_rate:.3f}{check}")
print(f"  hold rate: {m_base.hold_rate:.3f}{check}")

diffs = [i for i in test_ids if test_full[i].outcome != test_baseline[i].outcome]
print(
    f"decision-level differences between the two (same cost, different Python): {len(diffs)}{check}"
)
for i in diffs:
    print(
        f"  {i}: baseline {test_baseline[i].outcome} -> full pipeline {test_full[i].outcome} ({test_full[i].reason}){check}"
    )

matrix_labels = ["clear", "violates"]
matrix = [[0, 0], [0, 0]]
for gold_violates, held in zip(test_content_violates, full_held, strict=True):
    matrix[int(gold_violates)][int(held)] += 1
print(f"test confusion matrix, gold content by final decision{check}:")
print(f"  {'':<9s}  {'allowed':>9s}  {'held':>9s}{check}")
for row_label, row in zip(["gold clear", "gold violates"], matrix, strict=True):
    print(f"  {row_label:<9s}  {row[0]:>9d}  {row[1]:>9d}{check}")
plot_confusion_matrix(
    matrix, ["clear", "violates"], title=f"Test split, {len(test_ids)} messages{check}"
)

Neither variant misses more or fewer genuine disclosures: the adversarial message and the two business-rule misses are all confidently wrong (far from the gate's band) and in a thread of their own, so no mechanism this recipe builds catches them -- the confidence gate protects against genuine ambiguity, never against a confident mistake, the same limit recipe 15 names. What the full pipeline buys instead is entirely on the `false_hold_rate` side: `v14-thread`/`t14-thread` (clean on their own content, held only because their thread already had a violation) and `v16-uncertain`/`t16-uncertain` (a contact probability of 0.48, inside the gate's band) are held by the full pipeline and allowed by the baseline. Both are a deliberate trade: more messages wait for a person, for zero extra requests.

Nothing in this recipe is ever actually sent. Every `held` message from `test` goes into a simulated `ReviewQueue`; every `allowed` message is recorded to a simulated `ActionLog`, whose entries are always `executed: False` -- `jev_cookbook.simulation.ActionLog` never executes anything, which is what makes "nothing is sent" true in the code itself rather than only in prose.

In [ ]:
queue = ReviewQueue()
log = ActionLog()
for i in test_ids:
    result = test_full[i]
    if result.outcome == helpers.HELD:
        queue.submit(
            {"message_id": i, "verdicts": [v.rule + ":" + v.verdict for v in result.verdicts]},
            result.reason,
        )
    else:
        log.record("release", {"message_id": i}, rule=result.reason)

print(f"review queue: {len(queue)} held messages waiting{check}")
print(f"action log: {len(log)} allowed messages, every one executed=False{check}")
assert all(entry["executed"] is False for entry in log.to_dicts())

## What was and was not measured

Measured: that the pipeline runs end to end, and that the rule, the thread-escalation policy, the queue, and the evaluation behave as designed on 35 synthetic messages (32 scored, 3 shown above but never scored). Not measured: anything about how Jev performs, how fast it is, or what it costs. The stored answers in this offline run are written by hand as probabilities, some wrong on purpose -- including, on each split, a confidently fooled adversarial message and a genuine business-rule miss -- so every number above describes this fixture set, not a model. `allowed` means this pipeline found nothing to hold a message for; nothing in this recipe actually sends anything, in any outcome.

In [ ]:
if offline:
    print(f"Provenance: {backend.mode}. Not measured live: a pipeline check, not a Jev result.")
    print(f"N: {len(val_examples)} validation and {len(test_examples)} test messages{check}")
else:
    dates = ", ".join(getattr(backend, "recorded_dates", ()) or ()) or "this run"
    print(f"Provenance: {backend.mode}, model {backend.model}")
    print(f"Captured: {dates}")
    print(f"N: {len(val_examples)} validation and {len(test_examples)} test messages")

## Next steps

- Add a third sharing rule to `helpers.RULES` and `build_fixtures.py`, re-run `python build_fixtures.py`, and watch `pool_label_decisions` pool three propositions per message instead of two.
- Change `min_coverage` in "Python's part" and see how the shared confidence gate, and the message-level false-hold rate, move together.
- Add a third message to the `v-thread-a` thread in `build_fixtures.py` and confirm it is held purely by escalation, the same way `v14-thread` is above.
- Neighbouring recipes: [`15-sensitive-text-triage`](../15-sensitive-text-triage/) (the single-proposition Noul three-path pattern this recipe extends to a battery), [`38-abstention-and-review-queues`](../38-abstention-and-review-queues/) (thresholds chosen on `validation` and reported on `test`, in `Workflow & service operations`), and [`26-prompt-injection-screening`](../26-prompt-injection-screening/) (another adversarial-content case in `Trust & security`).